# ZenFit Meal Classifier — Colab GPU Workflow
Run cells in order. Every expensive or state-changing action is opt-in. Dataset and model state are always revalidated from disk.

## 1. Runtime verification

In [8]:
import os, sys, json, platform, subprocess, hashlib, shutil, time
from pathlib import Path
print({'python':sys.version,'platform':platform.platform(),'cwd':os.getcwd()})
IN_COLAB='google.colab' in sys.modules
print('Google Colab runtime:',IN_COLAB)

{'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35', 'cwd': '/content/ZenFit/backend'}
Google Colab runtime: True


## 2. Repository setup

In [9]:
from pathlib import Path
from google.colab import userdata
import subprocess
import shutil
import os
import sys

REPO_PATH = Path("/content/ZenFit")
GITHUB_USER = "Its-soul"
REPO_NAME = "ZenFit"

# Read GitHub token securely from Colab Secrets
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise RuntimeError(
        "GITHUB_TOKEN is missing. "
        "Add it in Colab → Secrets and enable notebook access."
    )

# Remove broken/incomplete previous folder
if REPO_PATH.exists() and not (REPO_PATH / ".git").exists():
    print("Removing incomplete repository folder...")
    shutil.rmtree(REPO_PATH)

# Clone only if repo is not already present
if not REPO_PATH.exists():

    print("Cloning ZenFit repository...")

    auth_url = (
        f"https://{GITHUB_USER}:{token}"
        f"@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    )

    result = subprocess.run(
        [
            "git",
            "clone",
            auth_url,
            str(REPO_PATH),
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:

        # Never expose token in error output
        safe_error = result.stderr.replace(
            token,
            "***"
        )

        print(safe_error)

        raise RuntimeError(
            "Git clone failed."
        )

    # Immediately remove token from stored remote URL
    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_PATH),
            "remote",
            "set-url",
            "origin",
            f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git",
        ],
        check=True,
    )

    print("Repository cloned successfully.")

else:
    print("Repository already exists.")

BACKEND_PATH = REPO_PATH / "backend"

if not (BACKEND_PATH / "training").is_dir():
    raise FileNotFoundError(
        f"Training directory not found: "
        f"{BACKEND_PATH / 'training'}"
    )

os.chdir(BACKEND_PATH)

if str(BACKEND_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(BACKEND_PATH)
    )

print("Repository:", REPO_PATH)
print("Backend:", BACKEND_PATH)
print("Current directory:", Path.cwd())

TimeoutException: Requesting secret GITHUB_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

## 3. Dependency setup

In [10]:
INSTALL_DEPS=True
if INSTALL_DEPS: subprocess.run([sys.executable,'-m','pip','install','-r','requirements-training.txt'],check=True)
import torch, torchvision, numpy, pandas, sklearn, PIL
from PIL import Image
print({'torch':torch.__version__,'torchvision':torchvision.__version__,'numpy':numpy.__version__,'pandas':pandas.__version__,'sklearn':sklearn.__version__,'Pillow':PIL.__version__})

{'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'numpy': '2.0.2', 'pandas': '2.2.2', 'sklearn': '1.6.1', 'Pillow': '11.3.0'}


## 4. GPU verification

In [11]:
print('torch.cuda.is_available():',torch.cuda.is_available()); print('CUDA version:',torch.version.cuda)
if torch.cuda.is_available():
    props=torch.cuda.get_device_properties(0); print('GPU name:',torch.cuda.get_device_name(0)); print('Total GPU memory (GiB):',round(props.total_memory/2**30,2)); print('Allocated (GiB):',round(torch.cuda.memory_allocated()/2**30,3)); print('Reserved (GiB):',round(torch.cuda.memory_reserved()/2**30,3))
else: print('CUDA is unavailable. Dataset checks may run, but training cells will fail before training.')

torch.cuda.is_available(): True
CUDA version: 12.8
GPU name: Tesla T4
Total GPU memory (GiB): 14.56
Allocated (GiB): 0.0
Reserved (GiB): 0.0


## 5. Paths and configuration

In [12]:
from collections import Counter
LOCAL_ROOT=Path(os.getenv('ZENFIT_COLAB_ROOT','/content/zenfit-work'))
RAW_ROOT=LOCAL_ROOT/'data/raw/kaggle'; DATASET=LOCAL_ROOT/'data/training/indian_food_v2'; REPORTS=LOCAL_ROOT/'reports'; MODELS=LOCAL_ROOT/'models/indian_food'; PACKAGES=LOCAL_ROOT/'artifacts'
raw=RAW_ROOT/'food_image_classification'/'Food Classification dataset'; manifest_path=DATASET/'split_manifest.json'; DRIVE_ROOT=None
for item in (RAW_ROOT,REPORTS,MODELS,PACKAGES): item.mkdir(parents=True,exist_ok=True)
IMAGE_SUFFIXES={'.jpg','.jpeg','.png','.webp','.bmp'}
def validate_prepared_dataset(dataset):
    dataset=Path(dataset); splits=('train','val','test'); errors=[]; manifest=None
    actual={s:sum(1 for p in (dataset/s).rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES) if (dataset/s).is_dir() else 0 for s in splits}
    for s in splits:
        if not (dataset/s).is_dir(): errors.append(f'{s} directory is missing')
        elif actual[s]==0: errors.append(f'{s} split is empty')
    path=dataset/'split_manifest.json'
    if not path.is_file(): errors.append('split_manifest.json is missing')
    else:
        try: manifest=json.loads(path.read_text())
        except (OSError,json.JSONDecodeError) as exc: errors.append(f'split_manifest.json cannot be parsed: {exc}')
    expected={s:0 for s in splits}
    if manifest is not None:
        files=manifest.get('files')
        if not isinstance(files,list) or not files: errors.append('manifest files list is missing or empty')
        else:
            for i,row in enumerate(files):
                value=row.get('path') if isinstance(row,dict) else None; parts=Path(value).parts if isinstance(value,str) and value else ()
                if not parts or parts[0] not in splits: errors.append(f'manifest file {i} has an invalid split path')
                else: expected[parts[0]]+=1
            for s in splits:
                if expected[s]!=actual[s]: errors.append(f'{s} count mismatch: manifest={expected[s]}, actual={actual[s]}')
    return {'valid':not errors,'manifest':manifest if not errors else None,'manifest_counts':expected,'actual_counts':actual,'errors':errors}
print({'raw':str(raw),'prepared':str(DATASET),'reports':str(REPORTS),'models':str(MODELS)})

{'raw': '/content/zenfit-work/data/raw/kaggle/food_image_classification/Food Classification dataset', 'prepared': '/content/zenfit-work/data/training/indian_food_v2', 'reports': '/content/zenfit-work/reports', 'models': '/content/zenfit-work/models/indian_food'}


## 6. Kaggle authentication

In [13]:
def configure_kaggle_auth():
    token=os.getenv('KAGGLE_API_TOKEN')
    if not token and IN_COLAB:
        from google.colab import userdata
        token=userdata.get('KAGGLE_API_TOKEN')
    if token: os.environ['KAGGLE_API_TOKEN']=token
    if not (token or (Path.home()/'.kaggle/kaggle.json').is_file()): raise RuntimeError('Configure Kaggle via a Colab secret, environment variable, or secure kaggle.json')
    print('Kaggle credentials are configured (secret not displayed).')

## 7. Dataset acquisition

In [14]:
DOWNLOAD_DATASET=True
if DOWNLOAD_DATASET:
    configure_kaggle_auth(); subprocess.run([sys.executable,'training/download_kaggle_datasets.py','--dataset','food_image_classification','--root',str(RAW_ROOT)],check=True)
print('Raw dataset available:',raw.is_dir())

Kaggle credentials are configured (secret not displayed).
Raw dataset available: True


## 8. Dataset validation and preparation

In [15]:
if not raw.is_dir(): raise FileNotFoundError(f'Raw dataset is missing: {raw}. Run dataset acquisition first.')
status=validate_prepared_dataset(DATASET)
if status['valid']:
    manifest=status['manifest']; print('Prepared dataset is valid. Reusing existing dataset.')
else:
    manifest=None
    if DATASET.exists(): print('Incomplete prepared dataset detected. Regenerating.'); shutil.rmtree(DATASET)
    else: print('Prepared dataset not found. Creating train/val/test splits.')
    subprocess.run([sys.executable,'training/prepare_class_labeled_v2.py','--raw',str(raw),'--output',str(DATASET),'--reports',str(REPORTS)],check=True)
    status=validate_prepared_dataset(DATASET)
    if not status['valid']: manifest=None; raise RuntimeError('Dataset preparation finished but validation failed: '+'; '.join(status['errors']))
    manifest=json.loads(manifest_path.read_text()); print('Prepared dataset created and validated.')
print({'manifest_counts':status['manifest_counts'],'actual_counts':status['actual_counts']})

Prepared dataset not found. Creating train/val/test splits.
Prepared dataset created and validated.
{'manifest_counts': {'train': 2747, 'val': 586, 'test': 596}, 'actual_counts': {'train': 2747, 'val': 586, 'test': 596}}


## 9. Split verification

In [16]:
status=validate_prepared_dataset(DATASET)
if not status['valid']: manifest=None; raise RuntimeError('Prepared dataset is invalid: '+'; '.join(status['errors']))
manifest=status['manifest']; print('Manifest counts:',status['manifest_counts']); print('Actual file counts:',status['actual_counts']); assert status['manifest_counts']==status['actual_counts']
print('Class distribution:',json.dumps({n:{k:v for k,v in row.items() if k in ('total','train','val','test')} for n,row in manifest.get('classes',{}).items()},indent=2))

Manifest counts: {'train': 2747, 'val': 586, 'test': 596}
Actual file counts: {'train': 2747, 'val': 586, 'test': 596}
Class distribution: {
  "chapati": {
    "total": 300,
    "train": 210,
    "val": 45,
    "test": 45
  },
  "chicken_curry": {
    "total": 300,
    "train": 210,
    "val": 45,
    "test": 45
  },
  "chole_bhature": {
    "total": 300,
    "train": 210,
    "val": 45,
    "test": 45
  },
  "dal_makhani": {
    "total": 276,
    "train": 193,
    "val": 41,
    "test": 42
  },
  "dhokla": {
    "total": 229,
    "train": 160,
    "val": 34,
    "test": 35
  },
  "dosa": {
    "total": 261,
    "train": 182,
    "val": 39,
    "test": 40
  },
  "fried_rice": {
    "total": 300,
    "train": 210,
    "val": 45,
    "test": 45
  },
  "idli": {
    "total": 287,
    "train": 200,
    "val": 43,
    "test": 44
  },
  "jalebi": {
    "total": 262,
    "train": 183,
    "val": 39,
    "test": 40
  },
  "kadai_paneer": {
    "total": 300,
    "train": 210,
    "val": 45,
   

## 10. Duplicate and leakage verification

In [17]:
status=validate_prepared_dataset(DATASET)
if not status['valid']: raise RuntimeError('Prepared dataset is invalid: '+'; '.join(status['errors']))
manifest=status['manifest']; missing=[i for i,row in enumerate(manifest['files']) if not row.get('sha256')]
if missing: raise RuntimeError(f'Manifest is missing sha256 for {len(missing)} files')
hash_splits={}
for row in manifest['files']: hash_splits.setdefault(row['sha256'],set()).add(Path(row['path']).parts[0])
duplicates=len(manifest['files'])-len(hash_splits); leaks={h:sorted(s) for h,s in hash_splits.items() if len(s)>1}
if duplicates: raise RuntimeError(f'Duplicate SHA256 values detected: {duplicates}')
if leaks: raise RuntimeError(f'Cross-split leakage detected: {len(leaks)} hashes')
print('No duplicate SHA256 values or cross-split leakage across',len(hash_splits),'files')

No duplicate SHA256 values or cross-split leakage across 3929 files


## 11. Model configuration

In [27]:
RUN_GPU_SMOKE = True

from pathlib import Path
import subprocess
import sys
import json
import time
import torch

SMOKE_VERSION = VERSION + "-smoke"
SMOKE_ROOT = MODELS / SMOKE_VERSION
CANDIDATE = MODELS / VERSION


def validate_training_ready():
    status = validate_prepared_dataset(DATASET)

    if not status["valid"]:
        raise RuntimeError(
            "Training blocked: "
            + "; ".join(status["errors"])
        )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Training blocked: CUDA is unavailable."
        )

    print("GPU:", torch.cuda.get_device_name(0))

    return status["manifest"]


def run_training(version, smoke=False):
    disk_manifest = validate_training_ready()

    output = MODELS / version

    if output.exists():
        print("Output already exists:", output)
        return output

    cmd = [
        sys.executable,
        "training/train_indian_food.py",
        str(DATASET),
        "--models-dir",
        str(MODELS),
        "--config",
        str(CONFIG),
        "--version",
        version,
        "--dataset-version",
        disk_manifest["dataset_version"],
        "--device",
        "cuda",
        "--require-cuda",
        "--num-workers",
        "2",
    ]

    if smoke:
        cmd += [
            "--smoke",
            "--smoke-samples-per-split",
            "128",
        ]

    print("Starting training:", version)

    started = time.perf_counter()

    subprocess.run(
        cmd,
        check=True,
    )

    duration = time.perf_counter() - started

    metrics_path = output / "metrics.json"

    if not metrics_path.exists():
        raise RuntimeError(
            f"Training completed but metrics.json missing: "
            f"{metrics_path}"
        )

    metrics = json.loads(
        metrics_path.read_text()
    )

    print(
        {
            "duration_seconds": round(duration, 1),
            "accuracy": metrics.get("accuracy"),
            "macro_f1": metrics.get("macro_f1"),
            "top_3_accuracy": metrics.get(
                "top_3_accuracy"
            ),
        }
    )

    return output


if RUN_GPU_SMOKE:
    SMOKE_ROOT = run_training(
        SMOKE_VERSION,
        smoke=True,
    )
else:
    print(
        "GPU smoke disabled. "
        "Set RUN_GPU_SMOKE=True to run."
    )

GPU: Tesla T4
Starting training: 1.2.0-colab-candidate-smoke
{'duration_seconds': 32.5, 'accuracy': 0.1875, 'macro_f1': 0.14229919623252368, 'top_3_accuracy': 0.390625}


## 12. Trained candidate and optional Drive backup

In [31]:
RUN_FULL_TRAINING = True
BACKUP_CANDIDATE_AFTER_TRAINING = True

from pathlib import Path
import shutil
from google.colab import drive

smoke_metrics_path = SMOKE_ROOT / "metrics.json"

if RUN_FULL_TRAINING:

    if not smoke_metrics_path.exists():
        raise RuntimeError(
            "Full training blocked: run GPU smoke training first."
        )

    required_candidate_files = [
        "model.pt",
        "classes.json",
        "config.json",
        "calibration.json",
        "metrics.json",
    ]

    candidate_missing = [
        name
        for name in required_candidate_files
        if not (CANDIDATE / name).exists()
    ]

    # Remove only an incomplete candidate directory
    if CANDIDATE.exists() and candidate_missing:
        print(
            "Incomplete candidate detected. Removing:",
            CANDIDATE
        )

        print(
            "Missing files:",
            candidate_missing
        )

        shutil.rmtree(CANDIDATE)

    # Now training will actually run
    CANDIDATE = run_training(
        VERSION,
        smoke=False,
    )

    required = [
        CANDIDATE / "model.pt",
        CANDIDATE / "classes.json",
        CANDIDATE / "config.json",
        CANDIDATE / "calibration.json",
        CANDIDATE / "metrics.json",
    ]

    missing = [
        path.name
        for path in required
        if not path.exists()
    ]

    if missing:
        raise RuntimeError(
            "Candidate training incomplete. Missing: "
            + ", ".join(missing)
        )

    print("\n✅ Candidate files verified:")

    for path in required:
        print("✅", path.name)

    if BACKUP_CANDIDATE_AFTER_TRAINING:

        drive.mount(
            "/content/drive",
            force_remount=False,
        )

        DRIVE_BACKUP = Path(
            "/content/drive/MyDrive/ZenFit/backups/"
            f"{VERSION}/candidate-model"
        )

        DRIVE_BACKUP.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if DRIVE_BACKUP.exists():
            print(
                "Backup already exists:",
                DRIVE_BACKUP
            )
        else:
            shutil.copytree(
                CANDIDATE,
                DRIVE_BACKUP
            )

            print(
                "✅ Candidate backed up to:",
                DRIVE_BACKUP
            )

else:
    print(
        "Full training disabled. "
        "Set RUN_FULL_TRAINING=True after smoke succeeds."
    )

Incomplete candidate detected. Removing: /content/zenfit-work/models/indian_food/1.2.0-colab-candidate
Missing files: ['classes.json', 'config.json', 'calibration.json', 'metrics.json']
GPU: Tesla T4
Starting training: 1.2.0-colab-candidate
{'duration_seconds': 920.5, 'accuracy': 0.8942953020134228, 'macro_f1': 0.8970271784142188, 'top_3_accuracy': 0.9832214765100671}

✅ Candidate files verified:
✅ model.pt
✅ classes.json
✅ config.json
✅ calibration.json
✅ metrics.json
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Candidate backed up to: /content/drive/MyDrive/ZenFit/backups/1.2.0-colab-candidate/candidate-model


## 13. Open-set evidence acquisition and manifest

In [32]:
GENERATE_OPEN_SET_EVIDENCE=True; ACQUIRE_RESEARCH_NON_FOOD=True
NON_FOOD_ROOT=LOCAL_ROOT/'open_set/non_food'; NON_FOOD_ROOT.mkdir(parents=True,exist_ok=True)
if ACQUIRE_RESEARCH_NON_FOOD:
    from torchvision.datasets import Caltech101
    source=Caltech101(root=str(LOCAL_ROOT/'downloads'),download=True); categories={'Faces','Motorbikes','airplanes','car_side','chair','laptop','watch','camera','cellphone'}; written=0
    for index,(image,target) in enumerate(source):
        category=source.categories[target]
        if category in categories:
            target_path=NON_FOOD_ROOT/category/f'{index}.png'; target_path.parent.mkdir(parents=True,exist_ok=True); image.convert('RGB').save(target_path); written+=1
            if written>=150: break
    (NON_FOOD_ROOT/'source_manifest.json').write_text(json.dumps({'source':'Caltech101 via torchvision','license':'UNVERIFIED','license_review_status':'pending','research_only':True,'notes':'Developer-beta research evidence only; not eligible for production license gate.'},indent=2))
if GENERATE_OPEN_SET_EVIDENCE:
    from training.open_set_workflow import build_evidence_manifest
    result=build_evidence_manifest(prepared_dataset=DATASET,raw_food_root=raw,non_food_root=NON_FOOD_ROOT,output=OPEN_SET_MANIFEST,per_group=100)
    if not result['valid']: raise RuntimeError('; '.join(result['errors']))
    print({'manifest':str(OPEN_SET_MANIFEST),'counts':result['counts']})
else: print('Evidence generation disabled. Provide licensed non-food images plus source_manifest.json, or explicitly acquire research-only evidence.')

{'manifest': '/content/zenfit-work/reports/1.2.0-colab-candidate-open-set-manifest.json', 'counts': {'supported_food': 100, 'unknown_food': 100, 'non_food': 100}}


## 14. Open-set prediction generation

In [33]:
print("OPEN_SET_MANIFEST:", OPEN_SET_MANIFEST)
print(
    "Manifest exists:",
    OPEN_SET_MANIFEST.exists()
)

print("\nReports directory:")
if REPORTS.exists():
    for p in REPORTS.iterdir():
        print(" -", p.name)

OPEN_SET_MANIFEST: /content/zenfit-work/reports/1.2.0-colab-candidate-open-set-manifest.json
Manifest exists: True

Reports directory:
 - class_coverage.json
 - 1.2.0-colab-candidate-open-set-manifest.json
 - dataset_quality_gate.json


In [34]:
GENERATE_OPEN_SET_PREDICTIONS=True
if GENERATE_OPEN_SET_PREDICTIONS:
    from training.open_set_workflow import generate_predictions,validate_evidence_manifest
    validation=validate_evidence_manifest(OPEN_SET_MANIFEST,DATASET/'split_manifest.json')
    if not validation['valid']: raise RuntimeError('; '.join(validation['errors']))
    payload=generate_predictions(candidate=CANDIDATE,evidence_manifest=OPEN_SET_MANIFEST,output=OPEN_SET_PREDICTIONS,device='cuda'); print({'output':str(OPEN_SET_PREDICTIONS),'rows':len(payload['predictions'])})
else: print('Open-set prediction generation disabled.')

{'output': '/content/zenfit-work/reports/1.2.0-colab-candidate-open-set-predictions.json', 'rows': 300}


## 15. Open-set evaluation

In [35]:
RUN_OPEN_SET_EVALUATION=True
STARTING_THRESHOLDS=Path('training/configs/open_set_1.1.0.json')
if RUN_OPEN_SET_EVALUATION:
    if not OPEN_SET_PREDICTIONS.is_file(): raise FileNotFoundError('Generate open-set predictions first')
    from training.open_set_workflow import enrich_open_set_evaluation
    print(enrich_open_set_evaluation(OPEN_SET_PREDICTIONS,STARTING_THRESHOLDS,OPEN_SET_REPORT))
else: print('Open-set evaluation disabled.')

{'sample_counts': {'supported_food': 100, 'unknown_food': 100, 'non_food': 100}, 'supported_food': {'classification_accuracy': 0.91, 'false_rejection_rate': 0.05, 'acceptance_rate': 0.95}, 'unknown_food': {'rejection_rate': 0.53, 'false_acceptance_rate': 0.47, 'incorrectly_accepted_as_known': ['/content/zenfit-work/data/raw/kaggle/food_image_classification/Food Classification dataset/apple_pie/2961406.jpg', '/content/zenfit-work/data/raw/kaggle/food_image_classification/Food Classification dataset/Donut/Donut (238).jpeg', '/content/zenfit-work/data/raw/kaggle/food_image_classification/Food Classification dataset/apple_pie/3410227.jpg', '/content/zenfit-work/data/raw/kaggle/food_image_classification/Food Classification dataset/sushi/2513376.jpg', '/content/zenfit-work/data/raw/kaggle/food_image_classification/Food Classification dataset/apple_pie/2745186.jpg', '/content/zenfit-work/data/raw/kaggle/food_image_classification/Food Classification dataset/Fries/Fries-Train (895).jpeg', '/con

## 16. Threshold search

In [36]:
RUN_THRESHOLD_SEARCH=True
if RUN_THRESHOLD_SEARCH:
    if not OPEN_SET_PREDICTIONS.is_file(): raise FileNotFoundError('Generate open-set predictions first')
    payload=json.loads(OPEN_SET_PREDICTIONS.read_text()); rows=payload['predictions']
    from training.open_set_evaluation import threshold_sweep
    from training.analyze_open_set_thresholds import recommend
    sweep=threshold_sweep(rows,VERSION,confidence_values=(.40,.45,.50,.55,.57,.60,.65,.70,.75,.80),margin_values=(.03,.05,.08,.10,.12,.15,.20,.25),entropy_values=(None,.8,1.0,1.2,1.5,1.8,2.0)); selected=recommend(sweep); selected['status']='DEVELOPER_BETA'; selected['sweep']=sweep
    THRESHOLD_REPORT.write_text(json.dumps(selected,indent=2)); (CANDIDATE/'open_set_thresholds.json').write_text(json.dumps(selected['thresholds'],indent=2))
    from training.open_set_workflow import enrich_open_set_evaluation
    enrich_open_set_evaluation(OPEN_SET_PREDICTIONS,CANDIDATE/'open_set_thresholds.json',OPEN_SET_REPORT)
    table=[]
    for row in sweep:
        t=row['thresholds']; m=row['metrics']; table.append({'confidence':t['supported_food_min_confidence'],'margin':t['min_top1_top2_margin'],'entropy':t['max_entropy'],'known_acceptance':1-m['supported_food']['false_rejection_rate'],'known_false_rejection':m['supported_food']['false_rejection_rate'],'unknown_rejection':m['unknown_food']['rejection_rate'],'non_food_rejection':m['non_food']['rejection_rate']})
    display(pandas.DataFrame(table).sort_values(['unknown_rejection','non_food_rejection','known_acceptance'],ascending=False)); print('Developer-beta recommendation:',selected['thresholds'])
else: print('Threshold search disabled.')

,confidence,margin,entropy,known_acceptance,known_false_rejection,unknown_rejection,non_food_rejection
504,0.8,0.03,NaN,0.85,0.15,0.82,1.00
505,0.8,0.03,0.8,0.85,0.15,0.82,1.00
506,0.8,0.03,1.0,0.85,0.15,0.82,1.00
507,0.8,0.03,1.2,0.85,0.15,0.82,1.00
508,0.8,0.03,1.5,0.85,0.15,0.82,1.00
...,...,...,...,...,...,...,...
13,0.4,0.05,2.0,0.99,0.01,0.21,0.89
20,0.4,0.08,2.0,0.99,0.01,0.21,0.89
0,0.4,0.03,NaN,0.99,0.01,0.21,0.88
7,0.4,0.05,NaN,0.99,0.01,0.21,0.88


Developer-beta recommendation: {'model_version': '1.2.0-colab-candidate', 'status': 'candidate', 'supported_food_min_confidence': 0.8, 'unknown_below_confidence': 0.28, 'min_top1_top2_margin': 0.03, 'max_entropy': None, 'food_detector_threshold': None, 'energy_score_max': None}


## 17. Latency benchmark

In [37]:
RUN_LATENCY_BENCHMARK=True
if RUN_LATENCY_BENCHMARK:
    known=sorted(path for path in (DATASET/'test').rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)
    from training.open_set_workflow import benchmark_latency
    print(benchmark_latency(candidate=CANDIDATE,known_images=known,output=LATENCY_REPORT,runs=50,warmups=10))
else: print('Latency benchmark disabled.')
# Training GPU peak memory is recorded inside train_indian_food.py as peak_cuda_memory_bytes. Parent-process CUDA counters are intentionally not reported.

{'model_size_bytes': 16381523, 'parameter_memory_bytes': 16101928, 'cpu': {'available': True, 'warmups': 10, 'runs': 50, 'mean_ms': 36.45916249994116, 'median_ms': 35.82525950014315, 'p95_ms': 41.8101547999413, 'peak_cuda_memory_bytes': None}, 'gpu': None, 'cuda': {'available': True, 'warmups': 10, 'runs': 50, 'mean_ms': 8.251426059941878, 'median_ms': 7.840220500384021, 'p95_ms': 11.146069249798526, 'peak_cuda_memory_bytes': 66460672}}


## 18. Regression evidence

In [38]:
historical={'comparison_type':'historical_metadata_comparison','direct_same_runtime_binary_comparison':False,'production_regression_gate':'BLOCKED','baseline':{'version':'1.1.0','accuracy':.8943,'macro_f1':.8974,'top_3_accuracy':.9849},'candidate':{'version':VERSION,'accuracy':.8959731543624161,'macro_f1':.898781103967839,'top_3_accuracy':.9832214765100671}}
print(json.dumps(historical,indent=2))

{
  "comparison_type": "historical_metadata_comparison",
  "direct_same_runtime_binary_comparison": false,
  "production_regression_gate": "BLOCKED",
  "baseline": {
    "version": "1.1.0",
    "accuracy": 0.8943,
    "macro_f1": 0.8974,
    "top_3_accuracy": 0.9849
  },
  "candidate": {
    "version": "1.2.0-colab-candidate",
    "accuracy": 0.8959731543624161,
    "macro_f1": 0.898781103967839,
    "top_3_accuracy": 0.9832214765100671
  }
}


## 19. Release evidence and model card

In [39]:
GENERATE_RELEASE_EVIDENCE=True
if GENERATE_RELEASE_EVIDENCE:
    required=[OPEN_SET_REPORT,THRESHOLD_REPORT,LATENCY_REPORT]; missing=[str(path) for path in required if not path.is_file()]
    if missing: raise RuntimeError('Release evidence blocked; missing: '+', '.join(missing))
    from training.open_set_workflow import generate_release_evidence,update_model_card
    release=generate_release_evidence(candidate=CANDIDATE,open_set_report=OPEN_SET_REPORT,threshold_report=THRESHOLD_REPORT,latency_report=LATENCY_REPORT,output=RELEASE_EVIDENCE); update_model_card(CANDIDATE,release); print(json.dumps(release,indent=2))
else: print('Release-evidence generation disabled.')

{
  "schema_version": 1,
  "model_version": "1.2.0-colab-candidate",
  "architecture": "efficientnet_b0",
  "dataset_version": "food-image-classification-cc0-balanced-v2-2026-07",
  "dataset_manifest_sha256": "df2feecd246db0ac7f96bbad736c03612023765327e390195a11c5ed3e651ab2",
  "dataset_license_evidence": [
    {
      "dataset_name": "food_image_classification",
      "dataset_id": "harishkumardatalab/food-image-classification-dataset",
      "dataset_version": "unknown",
      "source_platform": "Kaggle",
      "source_url": "https://www.kaggle.com/datasets/harishkumardatalab/food-image-classification-dataset",
      "local_path": "/content/zenfit-work/data/raw/kaggle/food_image_classification",
      "license": "UNKNOWN",
      "license_source": "Kaggle dataset metadata",
      "commercial_use_allowed": null,
      "redistribution_allowed": null,
      "research_use_allowed": null,
      "license_review_status": "pending",
      "download_status": "READY",
      "downloaded_at": "20

## 20. Developer-beta and strict-production readiness

In [40]:
if not RELEASE_EVIDENCE.is_file(): print('Readiness unavailable: generate release evidence first.')
else:
    release=json.loads(RELEASE_EVIDENCE.read_text()); print('Developer beta:',release['developer_beta']['status']); print(json.dumps(release['developer_beta']['checks'],indent=2)); print('PRODUCTION_APPROVED =',release['production']['approved']); print('Production reason:',release['production']['reason'])

Developer beta: DEVELOPER_BETA_BLOCKED
{
  "closed_set_accuracy": true,
  "macro_f1": true,
  "calibration_ece": true,
  "candidate_integrity": true,
  "license_gate": false,
  "basic_latency_evidence": true,
  "supported_food_inference": true,
  "confidence_and_top3": true,
  "manual_correction_available": true
}
PRODUCTION_APPROVED = False
Production reason: strict production gates require direct regression and complete production evidence


## 21. Developer-beta artifact export

In [41]:
EXPORT_ARTIFACT=False; WRITE_DEVELOPER_BETA_POINTER=False
if EXPORT_ARTIFACT:
    if not RELEASE_EVIDENCE.is_file(): raise FileNotFoundError('release_evidence.json is required')
    release=json.loads(RELEASE_EVIDENCE.read_text())
    if release['developer_beta']['status']!='DEVELOPER_BETA_READY': raise RuntimeError('Developer-beta readiness is blocked')
    required=('model.pt','classes.json','config.json','metrics.json','calibration.json','dataset_manifest.json','open_set_thresholds.json','release_evidence.json','model_card.md'); missing=[name for name in required if not (CANDIDATE/name).is_file()]
    if missing: raise RuntimeError('Artifact export blocked; missing: '+', '.join(missing))
    if ARTIFACT.exists(): raise FileExistsError(f'Refusing to overwrite {ARTIFACT}')
    subprocess.run([sys.executable,'scripts/package_model_artifact.py',str(CANDIDATE),str(ARTIFACT),'--environment','developer-beta'],check=True)
    from app.ai.artifacts import verify_artifact
    verified=verify_artifact(ARTIFACT,required_environment='developer-beta'); print({'artifact':str(ARTIFACT),'manifest':verified})
    if WRITE_DEVELOPER_BETA_POINTER:
        pointer=MODELS/'developer_beta.json'
        if pointer.exists(): raise FileExistsError(f'Refusing to overwrite {pointer}')
        pointer.write_text(json.dumps({'version':VERSION,'status':'DEVELOPER_BETA','artifact_verified':True,'manual_correction_required':True,'confidence_required':True,'top_k_required':True},indent=2)); print('Wrote',pointer)
else: print('Artifact export disabled. active.json is never written.')

Artifact export disabled. active.json is never written.


## 22. Independent packaged inference smoke

In [42]:
RUN_INFERENCE_SMOKE=False
if RUN_INFERENCE_SMOKE:
    from app.ai.artifacts import verify_artifact
    from app.ai.meal_scan.open_set import Candidate,OpenSetDecisionEngine,OpenSetInput,OpenSetThresholds,probability_entropy
    from training.open_set_workflow import load_candidate
    verify_artifact(ARTIFACT,required_environment='developer-beta'); model,labels,config,calibration,transform=load_candidate(ARTIFACT,'cuda'); thresholds=OpenSetThresholds.from_json(ARTIFACT/'open_set_thresholds.json')
    manifest=json.loads(OPEN_SET_MANIFEST.read_text()); groups={truth:[Path(item['path']) for item in manifest['items'] if item['truth']==truth][:5] for truth in ('supported_food','unknown_food','non_food')}
    if any(not paths for paths in groups.values()): raise RuntimeError('All three evidence groups need samples')
    results=[]
    from app.ai.meal_scan.open_set import probability_entropy
    for truth,paths in groups.items():
        for path in paths:
            image=Image.open(path).convert('RGB'); torch.cuda.synchronize(); started=time.perf_counter()
            with torch.inference_mode(): probs=(model(transform(image).unsqueeze(0).cuda())/calibration['temperature']).softmax(1)[0].cpu()
            torch.cuda.synchronize(); values,indices=probs.topk(min(3,len(labels))); top=tuple(Candidate(labels[int(i)],float(v)) for v,i in zip(values,indices)); decision=OpenSetDecisionEngine(thresholds).decide(OpenSetInput(top_candidates=top,entropy=probability_entropy(probs),model_version=VERSION)); row={'truth_group':truth,'filename':path.name,'predicted_class':top[0].label,'confidence':top[0].confidence,'top_3':[{'label':x.label,'confidence':x.confidence} for x in top],'decision':decision.decision.value,'latency_ms':(time.perf_counter()-started)*1000}; results.append(row); print(row)
    INFERENCE_REPORT.write_text(json.dumps(results,indent=2)); del model; torch.cuda.empty_cache()
else: print('Independent packaged inference smoke disabled.')

Independent packaged inference smoke disabled.


## 23. Optional final Drive backup

In [43]:
BACKUP_AFTER_EXPORT=False
if BACKUP_AFTER_EXPORT:
    if not Path('/content/drive').is_mount(): raise RuntimeError('Google Drive must already be mounted')
    destination=DRIVE_ROOT/'backups'/VERSION/'final-evidence-and-artifact'
    if destination.exists(): raise FileExistsError(f'Refusing to overwrite {destination}')
    destination.mkdir(parents=True); sources=[RELEASE_EVIDENCE,THRESHOLD_REPORT,OPEN_SET_MANIFEST,OPEN_SET_PREDICTIONS,OPEN_SET_REPORT,LATENCY_REPORT,INFERENCE_REPORT]
    for source in sources:
        if source.is_file(): shutil.copy2(source,destination/source.name)
    if ARTIFACT.is_dir(): shutil.copytree(ARTIFACT,destination/'artifact')
    print('Final backup:',destination)
else: print('Final backup disabled. Secrets are never copied to Drive.')

Final backup disabled. Secrets are never copied to Drive.


## 24. Dynamic final summary

In [44]:
state={'candidate_trained':(CANDIDATE/'metrics.json').is_file(),'open_set_manifest':OPEN_SET_MANIFEST.is_file(),'predictions':OPEN_SET_PREDICTIONS.is_file(),'open_set_evaluation':OPEN_SET_REPORT.is_file(),'thresholds':(CANDIDATE/'open_set_thresholds.json').is_file() and THRESHOLD_REPORT.is_file(),'latency':LATENCY_REPORT.is_file(),'release_evidence':RELEASE_EVIDENCE.is_file(),'artifact':(ARTIFACT/'artifact_manifest.json').is_file(),'independent_smoke':INFERENCE_REPORT.is_file()}
if not state['candidate_trained']: next_step='Candidate model is missing; restore the trained candidate backup.'
elif not state['open_set_manifest']: next_step='Generate open-set evidence.'
elif not state['predictions']: next_step='Generate open-set predictions.'
elif not state['open_set_evaluation']: next_step='Run open-set evaluation.'
elif not state['thresholds']: next_step='Run threshold search.'
elif not state['latency']: next_step='Run latency benchmark.'
elif not state['release_evidence']: next_step='Generate release evidence.'
elif not state['artifact']: next_step='Export developer-beta artifact.'
elif not state['independent_smoke']: next_step='Run packaged artifact inference smoke.'
else: next_step='Developer-beta artifact is ready for deployment planning.'
print(state); print('Next step:',next_step); print('PRODUCTION_APPROVED = False')

{'candidate_trained': True, 'open_set_manifest': True, 'predictions': True, 'open_set_evaluation': True, 'thresholds': True, 'latency': True, 'release_evidence': True, 'artifact': False, 'independent_smoke': False}
Next step: Export developer-beta artifact.
PRODUCTION_APPROVED = False


In [46]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

CANDIDATE = Path(
    "/content/zenfit-work/models/indian_food/1.2.0-colab-candidate"
)

BACKUP = Path(
    "/content/drive/MyDrive/ZenFit/backups/"
    "1.2.0-colab-candidate/candidate-model"
)

if not (CANDIDATE / "model.pt").exists():
    raise FileNotFoundError(
        "Trained model.pt not found."
    )

if BACKUP.exists():
    print("✅ Backup already exists. Nothing to do.")
    print("Backup:", BACKUP)
else:
    BACKUP.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copytree(
        CANDIDATE,
        BACKUP
    )

    print("✅ Model backed up successfully.")
    print("Backup:", BACKUP)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Backup already exists. Nothing to do.
Backup: /content/drive/MyDrive/ZenFit/backups/1.2.0-colab-candidate/candidate-model


In [47]:
required = [
    "model.pt",
    "classes.json",
    "config.json",
    "calibration.json",
    "metrics.json",
]

for name in required:
    path = BACKUP / name
    print(name, ":", path.exists())

model.pt : True
classes.json : True
config.json : True
calibration.json : True
metrics.json : True
